# InnerSource Pinning Example <a name="top"></a>

## Table of Contents
1. [Introduction](#introduction)
2. [End-to-end Pinning Example and Explanations](#example)
    - [Libraries and helper functions](#libraries)
    - [Local paths](#paths)
    - [2.1 Prepare input events](#inputevents)
    - [2.2 Define identifying attributes](#identifying-attributes)
    - [2.3 Design Near Neighbor function and Pairwise Match function](#nearneighbor-exactmatch)
        - [Near Neighbor function](#nearneighbor)
        - [Pairwise Match function](#exactmatch)
        - [Connected Components and pin stamping](#cc)
    - [2.4 Run pinning](#runpinning)
    - [2.5 Review pinning outputs](#output)
3. [Conclusion](#conclusion)

## Introduction <a name="introduction"></a>
Pinning is the process of grouping events belonging to same Entity.
<img src="./example/images/nb_01_intro.png" alt="Pinning Intro" width="700"/>

The Pinning process accepts curated events as input and progresses through three phases: Identity Extraction, Identity Pinning, and Heterogeneous Collection.
Initially, for each event, identity-related attributes are extracted to constitute an identity. We derive unique identities and their corresponding events to formulate an identity-event map.
Following this, identities associated with the same entities are clustered together in a process termed 'pinning.' This results in the formation of 'pins', representing individual entities along with their connected identities, leading to an establishment of a pin-identity map.
Finally, we join the pin-identity map with the identity-event map, thereby producing a complete mapping of pins and their associated events.

A Pinning job necessitates not only inputs but also user-defined identifying attributes, a Near Neighbor function, and a Pairwise (Exact) Match function to enable proper grouping of events. Detailed descriptions and examples of these files and functions are provided in subsequent sections.

<img src="./example/images/innersource_pinning_flowchart.png" alt="Pinning Flowchart" width="600"/>

[Return to Top](#top)

## End-to-end Pinning Example and Explanations <a name="example"></a>
The Pinning job orchestrates through the following steps:

1. [**Prepare input events**](#inputevents)
2. [**Define identifying attributes**](#identifying-attributes)
3. [**Design Near Neighbor function and Pairwise Match function**](#nearneighbor-exactmatch)
4. [**Run pinning**](#runpinning)
5. [**Review pinning outputs**](#output)


#### Libraries and helper functions <a name="libraries"></a>
First, we begin by importing necessary Python libraries and helper functions. These aid in extracting key information from inputs, intermediary results, and final outputs.

In [1]:
import os
import re
import json
import pandas as pd
pd.set_option('display.max_colwidth', None)

def print_jsonl_1stline(file):
    with open(file, 'r') as f:
        line = f.readline()
        data = json.loads(line)
        print(json.dumps(data, indent=2))

def retrieve_identifying_attributes(identifying_attributes_path):
    with open(identifying_attributes_path, 'r') as f:
        line = f.readline().strip()
        str_list = [s.strip()[1:-1] for s in line[1:-1].split(',')]
    return str_list
    
def retrieve_json_fields_input(file, fields):
    files = [file for file in os.listdir(input_folder) if file.endswith('.json')]
    patterns = [f'"{field}"\s*:\s*("[^"]*"|\d*)' for field in fields]
    values = []

    for file in files:
        with open(os.path.join(input_folder, file), 'r') as f:
            lines = f.readlines()
        for line in lines:
            if not line.strip():
                continue
            line_values = []
            for pattern in patterns:
                match = re.search(pattern, line)
                if match:
                    value = match.group(1)
                    if value.startswith('"') and value.endswith('"'):
                        value = value[1:-1]
                    line_values.append(value)
                else:
                    line_values.append(None)
            values.append(line_values)
    return values

def retrieve_json_fields_output(output_path, fields, array_field, array_element_field):
    result = []
    files = [f for f in os.listdir(output_path) if f.endswith('.json')]
    for file in files:
        with open(os.path.join(output_path, file), 'r') as f:
            for line in f:
                if not line.strip():
                    continue
                data = json.loads(line)
                line_dict = {}
                for field in fields:
                    if field in data:
                        line_dict[field] = data[field]
                if array_field in data and isinstance(data[array_field], list):
                    line_dict[array_element_field] = []
                    for element in data[array_field]:
                        if array_element_field in element:
                            line_dict[array_element_field].append(element[array_element_field])
                        else:
                            line_dict[array_element_field].append(None)
                result.append(line_dict)
    return result

def retrieve_json_fields_nearneighbor(nearneighbor_path, fields, array_field, array_element_field_sublevel_0, array_element_field_sublevel_1, array_element_fields_sublevel_2, element_field_sublevel_1):
    result = []
    field_renamed = 'nearneighborkey'
    array_field_renamed = 'identities'
    identityIds_field = 'identityIds'
    
    files = [f for f in os.listdir(nearneighbor_path) if f.endswith('.json')]
    for file in files:
        with open(os.path.join(nearneighbor_path, file), 'r') as f:
            for line in f:
                if not line.strip():
                    continue
                data = json.loads(line)
                line_dict = {}
                for field in fields:
                    if field in data:
                        line_dict[field_renamed] = data[field]
                if array_field in data and isinstance(data[array_field], list):
                    line_dict[array_field_renamed] = []
                    line_dict[identityIds_field] = []
                    for element in data[array_field]:
                        if array_element_field_sublevel_0 in element and isinstance(element[array_element_field_sublevel_0], dict):
                            if array_element_field_sublevel_1 in element[array_element_field_sublevel_0] and isinstance(element[array_element_field_sublevel_0][array_element_field_sublevel_1], dict):
                                element_dict = {}
                                attributes = []
                                for field in array_element_fields_sublevel_2:
                                    attributes.append(element[array_element_field_sublevel_0][array_element_field_sublevel_1].get(field, None))
                                element_dict[array_element_field_sublevel_0] = attributes
                            line_dict[array_field_renamed].append(element_dict)
                            if element_field_sublevel_1 in element[array_element_field_sublevel_0]:
                                line_dict[identityIds_field].append(element[array_element_field_sublevel_0][element_field_sublevel_1])
                result.append(line_dict)
    return result

def retrieve_json_fields_pinidentitymap(pinidentitymap_path, fields, array_field, array_element_field_sublevel_0, array_element_field_sublevel_1, array_element_fields_sublevel_2, element_field_sublevel_1):
    result = []
    
    files = [f for f in os.listdir(pinidentitymap_path) if f.endswith('.json')]
    for file in files:
        with open(os.path.join(pinidentitymap_path, file), 'r') as f:
            for line in f:
                if not line.strip():
                    continue
                data = json.loads(line)
                line_dict = {}
                for field in fields:
                    if field in data:
                        line_dict[field] = data[field]
                if array_field in data:
                    element = data[array_field]
                    if array_element_field_sublevel_0 in element and isinstance(element[array_element_field_sublevel_0], dict):
                        if array_element_field_sublevel_1 in element[array_element_field_sublevel_0] and isinstance(element[array_element_field_sublevel_0][array_element_field_sublevel_1], dict):
                            element_dict = {}
                            attributes = []
                            for field in array_element_fields_sublevel_2:
                                attributes.append(element[array_element_field_sublevel_0][array_element_field_sublevel_1].get(field, None))
                            element_dict[array_element_field_sublevel_0] = attributes
                        line_dict[array_element_field_sublevel_0] = attributes
                        if element_field_sublevel_1 in element[array_element_field_sublevel_0]:
                            line_dict[element_field_sublevel_1] = element[array_element_field_sublevel_0][element_field_sublevel_1]
                result.append(line_dict)
    return result

def filter_files(files, substring):
    for file in files:
        if substring in file:
            return file

def print_jsonl_by_pin(file, pin):
    with open(file, 'r') as f:
        for line in f:
            data = json.loads(line)
            if data.get("pin") == pin:
                print(json.dumps(data, indent=2))
                break

[Return to Top](#top)

#### Local paths <a name="paths"></a>
Subsequently, please revise the path variables to ensure they correctly point to the respective paths of the input, output, and identifying attributes configuration file on your machine.

In [2]:
input_folder = 'example/curated-events-input/events_batch_0'
output_root_path = 'example/pinned-output'
identifying_attributes_path = 'src/main/resources/identifying-attributes.json'

[Return to Top](#top)

### 1. Prepare input events <a name="inputevents"></a>
The inputs for the pinning process are curated events - events that have been validated, cleaned, and transformed into a fixed structure. 

Each event is assigned a unique "eventId", with all event details stored as values in various attribute fields.

We read from the inputs and display the first event as an example.

In [3]:
# Print the first input event as an example
files = [file for file in os.listdir(input_folder) if file.endswith('.json')]
file = os.path.join(input_folder, files[0])
print_jsonl_1stline(file)

{
  "lineage": {
    "dataSource": "230001",
    "receivedDate": 1485993600000,
    "fileName": "file:/Users/c65131a/Documents/Experian/oxygen-ri/oxygen-ri-data/E-230001.20161113.T",
    "subscriber": "230001",
    "fileAsOfDate": 1485993600000,
    "datasetId": "6755258703567384158",
    "recordHash": "f7476039e1ce1163bc0205d60e8f619f67cdcade"
  },
  "eventId": "-900",
  "eventData": {
    "ints": {},
    "strings": {
      "detailWarranty": "",
      "qualification": "D",
      "middleName": "RAMON",
      "email": "000000000000230001",
      "directionOfResidenceOrCommercialAddress": "0000243404220000000000000000000000000000Bogota",
      "departmentResidenceOrDepartmentCommercial": "",
      "directionOfCorrespondence": "",
      "nationalId": "009",
      "departmentLabor": "",
      "firstName": "NICOLAS",
      "firstLastName": "LOPEZ",
      "secondLastName": "CONTRERAS",
      "departmentOfCorrespondence": "",
      "cityLabor": "gota",
      "accountNumber": "0000001111166121

[Return to Top](#top)

### 2. Define identifying attributes <a name="identifying-attributes"></a>
From the comprehensive information contained within an event, we extract only those attributes related to the entity's identity. These attributes, termed 'identifying attributes', form an event's identity.

The identifying attributes are defined in a file as a list of field names. The path to this file is determined by *oxygen.stage.pinning.paths.identifyingAttributesFiles* in the configuration.

Below, we provide an example of the identifying attributes in use.

In [4]:
# Print identifying attributes
identifying_attributes = retrieve_identifying_attributes(identifying_attributes_path)
print("Identifying Attributes:")
print(identifying_attributes)

Identifying Attributes:
['firstName', 'middleName', 'firstLastName', 'secondLastName', 'nationalId', 'cellPhone']


The values of the identifying attributes are extracted from all the events and used to form their respective identities.

For instance, the identity of the event with the eventId "-900" is extracted as {"firstName": "NICOLAS", "middleName":"RAMON", "firstLastName":"LOPEZ", "secondLastName":"CONTRERAS", "nationalId":"009", "cellPhone":9513853222}.

Below, we display all the identities associated with the input events.

In [5]:
# Print identities in all events
additional_fields = ['eventId']
retrival_fields = identifying_attributes + additional_fields
retrived_values = retrieve_json_fields_input(input_folder, retrival_fields)
df_in = pd.DataFrame(retrived_values, columns=retrival_fields).set_index('eventId').sort_index()
df_in

,firstName,middleName,firstLastName,secondLastName,nationalId,cellPhone
eventId,,,,,,
-1000,MARTHA,MARIA,ALBA,PEREZ,010,2016264500
-700,MICHAEL,JEFFREY,JORDAN,,007,5882009999
-701,MICHAEL,JEFFREY,JORDAN,,007,5882009999
-702,MICHAEL,JEFFREY,JORDAN,,007,5882008888
-703,MICHAEL,,JORDAN,,007,5882007777
-704,MICHAEL,JEFFREY,JORDAN,,007,5882007777
-705,MIKE,JEFFREY,JORDAN,,007,5882008888
-706,MICHAEL,JEFFREY,JORDAN,,900,5882009999
-800,OMAR,CARLOS,ORTIZ,GARCIA,008,2016264500


[Return to Top](#top)

### 3. Design Near Neighbor function and Pairwise Match function <a name="nearneighbor-exactmatch"></a>
The Pairwise Match function is devised to distinguish identities related to different entities and authenticate those belonging to the same entities. This function is applied during pairwise comparisons. To sidestep the necessity for full pairwise comparisons, the complete identity space is subdivided into smaller neighborhoods suitable for parallel processing. A Near Neighbor function is designed such that identities sharing the same near neighbor key constitute a neighborhood. Within these neighborhoods, pairwise exact matches are executed on a considerably smaller scale.

Both the Near Neighbor and Pairwise Match functions are defined within the extended IdentityMatcher class, located at *src/main/scala/com/experian/edf/oxon/extension/ISIdentityMatcher.scala*.

Users can customize their own version of the extended IdentityMatcher using the *oxygen.stage.pinning.identityMatcher* configuration.

#### Near neighbor function <a name="nearneighbor"></a>
In this example, identities are segmented into smaller neighborhoods along three dimensions: nationalId, name construction (comprising the first 5 characters of the firstName and the first 5 characters of the firstLastName), and cellPhone. In each dimension, identities sharing an identical near neighbor key form a cluster.

Example near neighbor function:

<img src="./example/images/nb_03_nearneighbor.png" alt="Pinning Intro" width="1200"/>

In [6]:
# Print near neighber identity clusters from intermediary outputs
output_list = os.listdir(output_root_path)
nearneighbor_path = os.path.join(output_root_path, filter_files(output_list, '-pin-nearNeighbor'))
fields = ['id']
array_field = 'taggedEvents'
array_element_field_sublevel_0 = 'identity'
array_element_field_sublevel_1 = 'attributes'
array_element_fields_sublevel_2 = identifying_attributes
element_field_sublevel_1 = 'identityId'

nearneighbors = retrieve_json_fields_nearneighbor(nearneighbor_path, fields, array_field, array_element_field_sublevel_0, array_element_field_sublevel_1, array_element_fields_sublevel_2, element_field_sublevel_1)
df_nearneighbor = pd.DataFrame.from_dict(nearneighbors).set_index('nearneighborkey')
df_nearneighbor[['identityIds', 'identities']]

,identityIds,identities
nearneighborkey,,
008,"[6173067252208215900, -6003373127475236093]","[{'identity': ['OMAR', 'CARLOS', 'ORTIZ', 'GARCIA', '008', '2016264500']}, {'identity': ['OMAR', 'CARLOS', 'ORTIZ', 'GARCIA', '008', '98529440011']}]"
2016264500,"[8028546994782877476, 6173067252208215900]","[{'identity': ['MARTHA', 'MARIA', 'ALBA', 'PEREZ', '010', '2016264500']}, {'identity': ['OMAR', 'CARLOS', 'ORTIZ', 'GARCIA', '008', '2016264500']}]"
MARTHALBA,[8028546994782877476],"[{'identity': ['MARTHA', 'MARIA', 'ALBA', 'PEREZ', '010', '2016264500']}]"
9513853222,[2770401803749592803],"[{'identity': ['NICOLAS', 'RAMON', 'LOPEZ', 'CONTRERAS', '009', '9513853222']}]"
MICHAJORDA,"[521837037093398459, -1558426039705128230, 3075919223837439715, -2299841654561249412, 30516192875935665]","[{'identity': ['MICHAEL', '', 'JORDAN', '', '007', '5882007777']}, {'identity': ['MICHAEL', 'JEFFREY', 'JORDAN', '', '007', '5882009999']}, {'identity': ['MICHAEL', 'JEFFREY', 'JORDAN', '', '900', '5882009999']}, {'identity': ['MICHAEL', 'JEFFREY', 'JORDAN', '', '007', '5882008888']}, {'identity': ['MICHAEL', 'JEFFREY', 'JORDAN', '', '007', '5882007777']}]"
MIKEJORDA,[8780040121684966290],"[{'identity': ['MIKE', 'JEFFREY', 'JORDAN', '', '007', '5882008888']}]"
5882009999,"[-1558426039705128230, 3075919223837439715]","[{'identity': ['MICHAEL', 'JEFFREY', 'JORDAN', '', '007', '5882009999']}, {'identity': ['MICHAEL', 'JEFFREY', 'JORDAN', '', '900', '5882009999']}]"
007,"[8780040121684966290, 521837037093398459, -1558426039705128230, -2299841654561249412, 30516192875935665]","[{'identity': ['MIKE', 'JEFFREY', 'JORDAN', '', '007', '5882008888']}, {'identity': ['MICHAEL', '', 'JORDAN', '', '007', '5882007777']}, {'identity': ['MICHAEL', 'JEFFREY', 'JORDAN', '', '007', '5882009999']}, {'identity': ['MICHAEL', 'JEFFREY', 'JORDAN', '', '007', '5882008888']}, {'identity': ['MICHAEL', 'JEFFREY', 'JORDAN', '', '007', '5882007777']}]"
NICOLLOPEZ,[2770401803749592803],"[{'identity': ['NICOLAS', 'RAMON', 'LOPEZ', 'CONTRERAS', '009', '9513853222']}]"


[Return to Top](#top)

#### Pairwise Match function <a name="exactmatch"></a>
The Pairwise Match (or doesMatch) function compares two identities, returning 'true' if the identities belong to the same entity and 'false' otherwise. In the demonstrated implementation, if the nationalIds match, the function assigns a similarity score of 0.5. If both the firstName and firstLastName match, a score of 0.3 is awarded, and identical cellPhone numbers also receive a score of 0.3. When the cumulative similarity score 0.6 or larger, the two identities are deemed to belong to the same entity.

Example pairwise match (doesMatch) function:

<img src="./example/images/nb_04_doesmatch.png" alt="Pinning Intro" width="900"/>

With the returned Boolean values ('true' or 'false') from the user-defined Pairwise Match pairwise comparison, a Union Find algorithm maintains connected identities in the same clusters within a neighborhood. These clusters are referred to as pairwise matched clusters.

[Return to Top](#top)

#### Connected Components and pin stamping <a name="cc"></a>
Pairwise matched identity clusters are then merged across neighborhoods using the Connected Components algorithm. Each connected identity cluster is allocated a unique pin number that represents the entity. For example, pin 2003 contains a cluster of 2 connected identities, and pin 2001 has a cluster of 6 connected identities.

In [7]:
# Print pin-identity-map from intermediary outputs
pinidentitymap_path = os.path.join(output_root_path, filter_files(output_list, '-pin-identity-pin-map'))
fields = ['pin']
array_field = 'taggedIdentity'
array_element_field_sublevel_0 = 'identity'
array_element_field_sublevel_1 = 'attributes'
array_element_fields_sublevel_2 = identifying_attributes
element_field_sublevel_1 = 'identityId'

pinidentitymap = retrieve_json_fields_pinidentitymap(pinidentitymap_path, fields, array_field, array_element_field_sublevel_0, array_element_field_sublevel_1, array_element_fields_sublevel_2, element_field_sublevel_1)
df_pinidentitymap = pd.DataFrame.from_dict(pinidentitymap).set_index('pin').sort_index()
df_pinidentitymap[['identityId', 'identity']]


,identityId,identity
pin,,
2000,8028546994782877476,"[MARTHA, MARIA, ALBA, PEREZ, 010, 2016264500]"
2001,8780040121684966290,"[MIKE, JEFFREY, JORDAN, , 007, 5882008888]"
2001,521837037093398459,"[MICHAEL, , JORDAN, , 007, 5882007777]"
2001,-1558426039705128230,"[MICHAEL, JEFFREY, JORDAN, , 007, 5882009999]"
2001,3075919223837439715,"[MICHAEL, JEFFREY, JORDAN, , 900, 5882009999]"
2001,-2299841654561249412,"[MICHAEL, JEFFREY, JORDAN, , 007, 5882008888]"
2001,30516192875935665,"[MICHAEL, JEFFREY, JORDAN, , 007, 5882007777]"
2002,2770401803749592803,"[NICOLAS, RAMON, LOPEZ, CONTRERAS, 009, 9513853222]"
2003,6173067252208215900,"[OMAR, CARLOS, ORTIZ, GARCIA, 008, 2016264500]"


[Return to Top](#top)

### 4. Run pinning <a name="runpinning"></a>
Run pinning with sample input, configurations, Near Neighbor function, and Pairwise Match function.

In [8]:
# Uncomment the lines in this block and run pinning
# !cp /data/src/main/resources/log4j.properties /opt/spark/conf/;
# !spark-submit --master local[2] --driver-memory 2g --executor-memory 2g --conf "spark.driver.extraJavaOptions=-Dlog4j.configuration=log4j.properties" --class com.experian.edf.oxygen.pinning.app.PinningApp target/oxygen-pinning-innersource-1.0-jar-with-dependencies.jar env.conf pinning-application.conf;

[Return to Top](#top)

### 5. Review pinning outputs <a name="output"></a>
The final output of pinning undergoes a Heterogeneous Collection step, joining the pin-identity map with the identity-event map. Ultimately, the results consist of a pin-event map, with pins mapped to a list of their associated events.

In the example output, an entity represented by pin 2001 has events with eventIds "-700", "-701", "-702", "-703", "-704", "-705", and "-706". Another entity, represented by pin 2003, has events with eventIds "-800" and "-801". The remaining two entities, represented by pin 2000 and pin 2002, each have only one event.

In [9]:
# Verify output path
output_root_path = 'example/pinned-output'
# output_root_path = 'example/output-demo'
output_list = os.listdir(output_root_path)

# Print pin-eventId map from output
output_path = os.path.join(output_root_path, filter_files(output_list, '-pin-hc'))
fields = ['pin']
array_field = 'events'
array_element_field = 'eventId'

output = retrieve_json_fields_output(output_path, fields, array_field, array_element_field)
df_out = pd.DataFrame.from_dict(output).set_index('pin').sort_index()
df_out

,eventId
pin,
2000,[-1000]
2001,"[-700, -701, -702, -703, -704, -705, -706]"
2002,[-900]
2003,"[-800, -801]"


We can validate the output of pinned events against the original input of curated events to examine how the events, represented by eventIds, are aggregated. This is particularly applicable for pin 2001 and its associated events "-700", "-701", "-702", "-703", "-704", "-705", "-706". The individual, MICHAEL JORDAN, has numerous identities, incorporating three distinct cell phone numbers, two unique national IDs, and an alternative spelling of his first name. Nevertheless, through the pinning process, these events are cohesively grouped together.

In [10]:
# Print the input events and their corresponding identities again to cross-verify the pinned events in the pinning output. 
df_in

,firstName,middleName,firstLastName,secondLastName,nationalId,cellPhone
eventId,,,,,,
-1000,MARTHA,MARIA,ALBA,PEREZ,010,2016264500
-700,MICHAEL,JEFFREY,JORDAN,,007,5882009999
-701,MICHAEL,JEFFREY,JORDAN,,007,5882009999
-702,MICHAEL,JEFFREY,JORDAN,,007,5882008888
-703,MICHAEL,,JORDAN,,007,5882007777
-704,MICHAEL,JEFFREY,JORDAN,,007,5882007777
-705,MIKE,JEFFREY,JORDAN,,007,5882008888
-706,MICHAEL,JEFFREY,JORDAN,,900,5882009999
-800,OMAR,CARLOS,ORTIZ,GARCIA,008,2016264500


In [11]:
# Print one of the pins in the output
files = [file for file in os.listdir(output_path) if file.endswith('.json')]
for f in files:
    file = os.path.join(output_path, f)
    print_jsonl_by_pin(file, 2003)

{
  "pin": 2003,
  "events": [
    {
      "asOfDate": 1485993600000,
      "didCreateAsOfDateFromReceivedDate": false,
      "eventData": {
        "booleans": {
          "endMostMessage": false,
          "isNewSubscriberRelation": false
        },
        "complex": {
          "officeOfFiling": "{\"mainCity\":\"Bogota\",\"secondCity\":\"BUENAVENTURA\",\"number\":0}"
        },
        "longs": {
          "adjDate": 0,
          "adjective": 0,
          "ageOfMora": 0,
          "arrears": 0,
          "availableValue": 0,
          "balance": 0,
          "balanceValueInMora": 0,
          "cardNumber": -1,
          "cellPhone": 2016264500,
          "classCard": 0,
          "clausePermanence": 0,
          "contributionsCancelled": 0,
          "creditLimit": 0,
          "daneCityCodeCorrespondence": -1,
          "daneCityCodeFiling": -1,
          "daneCityCodeLabor": -1,
          "daneCityCodeResidenceOrDaneCityCodeCommercial": -1,
          "dateClausePermanence": 0,
  

[Return to Top](#top)

## Conclusion <a name="conclusion"></a>
Pinning is a process that associates events with the same entities. It extracts identity information from the events, reduces the comparison scope from the entire identity universe to small, multidimensional near neighbors for distributed computing, rigorously verifies identity connections within near neighbors through pairwise matching, and then merges identity connections across near neighbors. Unique pins are then assigned to connected identity clusters to represent individual entities. Ultimately, pins are assigned to the events associated with the connected identities.


[Return to Top](#top)